# 03 — A2A Protocol Basics
**Study Notebook 3 of 3 · Estimated time: 30-40 minutes**

By the end of this notebook you will:
- Understand the A2A (Agent-to-Agent) protocol: Agent Cards, Tasks, Artifacts
- Build 3 specialist A2A agents — the ones the final demo wires together
- See how one A2A agent (`OnboardingGuideAgent`) wraps a smolagents `CodeAgent` inside it
- *(Optional)* See what a real, over-the-network A2A server looks like — not just simulated in-process

This notebook stops at the agents themselves — deliberately. Wiring them into a LangGraph pipeline is
`foss_demo.ipynb`'s job; duplicating that wiring here would just be the same pipeline twice. Study the
agents in isolation here, then see them orchestrated for real in the demo.

```
What we're building -- three independent specialists:

  A2A: RepoFinderAgent         <- searches GitHub API
  A2A: ComplexityRaterAgent    <- rates repos for your level
  A2A: OnboardingGuideAgent    <- smolagents CodeAgent inside!
                                   fetches CONTRIBUTING.md + open issues
```

Parts 1-2 build these three agents using a small `A2AAgent` base class we write ourselves — this
mirrors the real protocol's shape (Agent Card + Task) without needing any servers or ports, so it works
identically anywhere. Part 3 is optional extra reading: it shows the same idea served for real over
HTTP using a small community package.

---
## Step 0 — Install & API Keys

In [ ]:
!pip install -q "smolagents[toolkit]" litellm langgraph requests

In [ ]:
import os

def load_key(name: str):
    if os.environ.get(name):
        print(f'{name} loaded from environment'); return
    try:
        from google.colab import userdata
        os.environ[name] = userdata.get(name)
        print(f'{name} loaded from Colab Secrets'); return
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
        if os.environ.get(name):
            print(f'{name} loaded from .env file'); return
    except ImportError:
        pass
    import getpass
    os.environ[name] = getpass.getpass(f'{name}: ')

load_key('GROQ_API_KEY')

In [ ]:
import requests, json, uuid, base64, re
from dataclasses import dataclass, field
from typing import Any, Optional, List
from enum import Enum
from smolagents import CodeAgent, LiteLLMModel, tool

model = LiteLLMModel(model_id="groq/llama-3.3-70b-versatile")

def gh_get(url, params=None):
    """GitHub API request with optional auth header."""
    headers = {}
    if os.environ.get("GITHUB_TOKEN"):
        headers["Authorization"] = f"token {os.environ['GITHUB_TOKEN']}"
    return requests.get(url, headers=headers, params=params, timeout=10)

print("Imports ready.")

---
## Part 1 — The A2A Protocol

**A2A (Agent-to-Agent)** is Google's open protocol (April 2025) for agent interoperability.

| Concept | What it is | Real-world analogy |
|---------|-----------|--------------------|
| **Agent Card** | JSON manifest at `/.well-known/agent.json` | A business card for your agent |
| **Task** | Structured message: message + context → artifacts | A work ticket |
| **Artifact** | Structured output from the agent | The completed deliverable |

In production: each agent runs as a separate HTTP service. Here: same interface, in-process, for Colab portability.

In [ ]:
class TaskState(Enum):
    SUBMITTED = "submitted"
    WORKING   = "working"
    COMPLETED = "completed"
    FAILED    = "failed"

@dataclass
class A2ATask:
    """Mirrors the A2A Task object sent between agents."""
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    message: str = ""
    context: dict = field(default_factory=dict)
    state: TaskState = TaskState.SUBMITTED
    artifacts: List[dict] = field(default_factory=list)
    error: Optional[str] = None

class A2AAgent:
    """
    Base class implementing the A2A protocol pattern.
    In production: subclasses run as independent HTTP services.
    The orchestrator calls POST /tasks with an A2ATask payload.
    """

    @property
    def agent_card(self) -> dict:
        """The A2A Agent Card — returned at /.well-known/agent.json in production."""
        raise NotImplementedError

    def send_task(self, message: str, context: dict = None) -> A2ATask:
        """Send a task to this agent. Mirrors HTTP POST /tasks."""
        task = A2ATask(message=message, context=context or {})
        print(f"  [A2A ->] {self.agent_card['name']} | task:{task.id}")
        task.state = TaskState.WORKING
        try:
            result = self._handle(task)
            task.artifacts = [{"type": "data", "data": result}]
            task.state = TaskState.COMPLETED
            print(f"  [A2A <-] {self.agent_card['name']} | task:{task.id} done")
        except Exception as e:
            task.error = str(e)
            task.state = TaskState.FAILED
            print(f"  [A2A x]  {self.agent_card['name']} | task:{task.id} failed: {e}")
        return task

    def _handle(self, task: A2ATask) -> Any:
        raise NotImplementedError

print("A2A base classes defined.")

---
## Part 2 — Build the 3 Specialist Agents

### Agent 1: RepoFinderAgent
Searches GitHub for repos matching a developer's skill profile.

In [ ]:
class RepoFinderAgent(A2AAgent):

    @property
    def agent_card(self):
        return {
            "name": "RepoFinderAgent",
            "description": "Discovers GitHub repositories matching a developer skill profile with open contribution opportunities",
            "version": "1.0.0",
            "capabilities": {"streaming": False, "push_notifications": False},
            "skills": [{
                "id": "find_matching_repos",
                "name": "Find Matching Repositories",
                "description": "Search GitHub for repos with good-first-issues matching given languages and level",
                "input_modes": ["text"], "output_modes": ["data"]
            }]
        }

    def _handle(self, task: A2ATask) -> list:
        languages  = task.context.get("languages", ["Python"])
        experience = task.context.get("experience", "intermediate")
        star_range = "100..2000" if experience == "beginner" else "500..50000"

        results = []
        for lang in languages[:2]:
            query = f"language:{lang} good-first-issues:>3 stars:{star_range}"
            resp  = gh_get(
                "https://api.github.com/search/repositories",
                params={"q": query, "sort": "updated", "per_page": 5}
            )
            if resp.status_code == 200:
                for repo in resp.json().get("items", []):
                    results.append({
                        "full_name":   repo["full_name"],
                        "description": (repo.get("description") or "")[:120],
                        "stars":       repo["stargazers_count"],
                        "language":    repo.get("language", ""),
                        "open_issues": repo["open_issues_count"],
                        "topics":      repo.get("topics", [])[:5]
                    })
        return results[:6]

# Show the Agent Card
print(json.dumps(RepoFinderAgent().agent_card, indent=2))

In [ ]:
task = RepoFinderAgent().send_task(
    message="Find repos for a Python developer",
    context={"languages": ["Python"], "experience": "beginner"}
)
print(f"State: {task.state.value}")
for repo in task.artifacts[0]["data"]:
    print(f"  {repo['full_name']} ({repo['stars']} stars) — {repo['description'][:60]}")

### Agent 2: ComplexityRaterAgent
Rates a repository's complexity using deterministic heuristics — no LLM call needed.

In [ ]:
class ComplexityRaterAgent(A2AAgent):

    @property
    def agent_card(self):
        return {
            "name": "ComplexityRaterAgent",
            "description": "Rates the contributor complexity of a GitHub repository",
            "version": "1.0.0",
            "capabilities": {"streaming": False, "push_notifications": False},
            "skills": [{
                "id": "rate_complexity",
                "name": "Rate Repository Complexity",
                "description": "Analyses repo structure and rates as beginner-friendly / intermediate / advanced",
                "input_modes": ["text"], "output_modes": ["data"]
            }]
        }

    def _handle(self, task: A2ATask) -> dict:
        owner = task.context["owner"]
        repo  = task.context["repo"]

        repo_data  = gh_get(f"https://api.github.com/repos/{owner}/{repo}").json()
        langs_data = gh_get(f"https://api.github.com/repos/{owner}/{repo}/languages").json()

        stars     = repo_data.get("stargazers_count", 0)
        forks     = repo_data.get("forks_count", 0)
        num_langs = len(langs_data) if isinstance(langs_data, dict) else 1

        score = 0
        if stars > 10000: score += 2
        elif stars > 2000: score += 1
        if num_langs > 4: score += 2
        elif num_langs > 2: score += 1
        if forks > 1000: score += 1

        if score <= 1:
            complexity, tip = "beginner-friendly", "Small codebase. Great starting point."
        elif score <= 3:
            complexity, tip = "intermediate", "Moderate complexity. Some experience helpful."
        else:
            complexity, tip = "advanced", "Large project. Better after a few contributions elsewhere."

        return {
            "repo":       f"{owner}/{repo}",
            "complexity": complexity,
            "tip":        tip,
            "stars":      stars,
            "languages":  list(langs_data.keys())[:5] if isinstance(langs_data, dict) else []
        }

task = ComplexityRaterAgent().send_task(
    message="Rate smolagents",
    context={"owner": "huggingface", "repo": "smolagents"}
)
print(json.dumps(task.artifacts[0]["data"], indent=2))

### Agent 3: OnboardingGuideAgent

This is where it gets interesting. This A2A agent **internally wraps a smolagents `CodeAgent`**.

```
LangGraph node
    └── calls OnboardingGuideAgent.send_task()  [A2A]
            └── runs CodeAgent.run()            [smolagents]
                    └── writes Python to call:
                            fetch_contributing_guide()  [tool]
                            fetch_starter_issues()      [tool]
                                └── GitHub API
```

In [ ]:
@tool
def fetch_contributing_guide(owner: str, repo: str) -> str:
    """Fetches the CONTRIBUTING.md from a GitHub repository.

    Args:
        owner: GitHub username or organisation.
        repo: Repository name.
    """
    for path in ["CONTRIBUTING.md", ".github/CONTRIBUTING.md", "docs/contributing.md"]:
        resp = gh_get(f"https://api.github.com/repos/{owner}/{repo}/contents/{path}")
        if resp.status_code == 200:
            raw = base64.b64decode(resp.json()["content"]).decode("utf-8", errors="ignore")
            clean = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', raw)
            clean = re.sub(r'[`#*<>!]', '', clean)
            return clean.strip()[:1200]
    return "No CONTRIBUTING.md found. Check the README for how to contribute."

@tool
def fetch_starter_issues(owner: str, repo: str) -> str:
    """Fetches open 'good first issue' and 'help wanted' issues from a GitHub repository.

    Args:
        owner: GitHub username or organisation.
        repo: Repository name.
    """
    found = []
    for label in ["good first issue", "help wanted"]:
        resp = gh_get(
            f"https://api.github.com/repos/{owner}/{repo}/issues",
            params={"labels": label, "state": "open", "per_page": 3}
        )
        if resp.status_code == 200:
            for issue in resp.json():
                if isinstance(issue, dict):
                    found.append(f"[{label}] #{issue['number']}: {issue['title']}")
    return "\n".join(found) if found else "No labelled starter issues found — browse open issues on GitHub."

print("Inner tools ready.")

In [ ]:
class OnboardingGuideAgent(A2AAgent):
    """An A2A agent that runs a smolagents CodeAgent internally."""

    def __init__(self, llm_model):
        self._inner_agent = CodeAgent(
            tools=[fetch_contributing_guide, fetch_starter_issues],
            model=llm_model
        )

    @property
    def agent_card(self):
        return {
            "name": "OnboardingGuideAgent",
            "description": "Generates a personalised first-contribution guide using an inner smolagents CodeAgent",
            "version": "1.0.0",
            "capabilities": {"streaming": False, "push_notifications": False},
            "skills": [{
                "id": "generate_onboarding",
                "name": "Generate Onboarding Guide",
                "description": "Fetches CONTRIBUTING.md and open issues, returns a step-by-step first-PR plan",
                "input_modes": ["text"], "output_modes": ["text"]
            }]
        }

    def _handle(self, task: A2ATask) -> str:
        owner  = task.context["owner"]
        repo   = task.context["repo"]
        skills = task.context.get("skills", ["Python"])

        # The CodeAgent writes Python to call the tools — watch it happen live
        return self._inner_agent.run(
            f"Generate a beginner-friendly first-contribution guide for {owner}/{repo}. "
            f"The developer knows: {', '.join(skills)}. "
            "Use the tools to fetch the contributing guide and starter issues. "
            "Return: 1) Setup steps (3 bullets), 2) Best first issue to tackle and why, "
            "3) Which files are likely to need editing."
        )

onboarding = OnboardingGuideAgent(model)

# Run it — watch the LLM write Python live
task = onboarding.send_task(
    message="Onboard me into smolagents",
    context={"owner": "huggingface", "repo": "smolagents", "skills": ["Python"]}
)
print("\n" + task.artifacts[0]["data"])

---
## Part 3 (Optional) — What Real A2A Looks Like

Everything above — `A2AAgent`, `send_task()` — is a simulation we wrote ourselves, on purpose: it
runs anywhere with zero extra infrastructure. In a real deployment, an A2A agent is an actual HTTP
server: it serves a real Agent Card and accepts real JSON-RPC 2.0 task requests.

This section shows that using [`a2a-smol-adapter`](https://pypi.org/project/a2a-smol-adapter/) — a
small community package (not an official Google or HuggingFace project; alpha, v0.1.0 at time of
writing) that wraps a smolagents `CodeAgent` as a real A2A server.

> **This part is genuinely optional.** Reading it is enough to understand the idea — running it starts
> a local server on a port, which not every environment allows.

In [ ]:
!pip install -q a2a-smol-adapter

In [ ]:
import threading, time
from a2a_smol_adapter import SmolA2AServer

# Reuse the same CodeAgent from OnboardingGuideAgent above -- this time served for real
real_server = SmolA2AServer(
    onboarding._inner_agent,
    name="onboarding-guide-agent",
    host="127.0.0.1",
    port=5001,
)

# server.run() blocks, so it runs on a background thread to keep the notebook usable
threading.Thread(target=real_server.run, daemon=True).start()
time.sleep(2)  # give the server a moment to start

print("Serving a REAL Agent Card at:")
print("  http://127.0.0.1:5001/.well-known/agent-card.json")

In [ ]:
import requests

# This is a real HTTP request -- the Agent Card is no longer just a Python dict
print(requests.get("http://127.0.0.1:5001/.well-known/agent-card.json").json())

In [ ]:
from a2a_smol_adapter import SmolA2ADelegateTool

# A tool that lets ANY smolagents agent call our real A2A server over HTTP
delegate = SmolA2ADelegateTool(remote_url="http://127.0.0.1:5001/")

caller_agent = CodeAgent(tools=[delegate], model=model)
result = caller_agent.run(
    "Ask the remote agent to generate a first-contribution guide for huggingface/smolagents "
    "for a developer who knows Python."
)
print(result)

**What actually changed vs. the simulation in Parts 1-2:**

| | Our `A2AAgent` (Parts 1-2) | `a2a-smol-adapter` (this section) |
|---|---|---|
| Transport | In-process Python call | Real HTTP, JSON-RPC 2.0 |
| Agent Card | A Python dict | Served at a real `/.well-known/agent-card.json` URL |
| Calling it | `agent.send_task(...)` | Another agent's tool makes a genuine network request |
| Where it can run | Anywhere, zero setup | Needs a listening port — here, a background thread |

Same idea, same interface shape (Agent Card + Task) — just genuinely over the wire instead of simulated.

> **A note on this package:** it's a small, single-maintainer, alpha-stage (v0.1.0) project — good for
> seeing what real A2A looks like in practice, not yet something to build a production system on.

---
## Summary

| Agent | What it does | LLM call? |
|-------|-------------|-----------|
| `RepoFinderAgent` | GitHub API search for matching repos | No |
| `ComplexityRaterAgent` | Python heuristic on repo metadata (stars, languages, forks) | No |
| `OnboardingGuideAgent` | Wraps a smolagents `CodeAgent` — fetches CONTRIBUTING.md + issues, drafts a guide | **Yes — once per repo** |

Each of these is a standalone A2A agent: same `agent_card` + `send_task()` interface, independent of
how it's wired up. Only `OnboardingGuideAgent` calls an LLM at all.

**Next:** `foss_demo.ipynb` — see all three wired into a LangGraph pipeline for the full FOSS
Contribution Matchmaker.